In [1]:
import sys
sys.path.append('../STACAME/')
import STACAME #.STACAME as STACAME
# the location of R (used for the mclust clustering)
import os
os.environ['R_HOME'] = "/data/zhanglab/zhangbiao/anaconda3/envs/env_stasage/lib/R"
os.environ['R_USER'] = "/data/zhanglab/zhangbiao/anaconda3/envs/env_stasage/lib/python3.11/site-packages/rpy2"
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri

import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import scipy.linalg
from scipy.sparse import csr_matrix
import pandas as pd
import torch
from STACAME.analysis import merge_embedding
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score
import seaborn as sns

#from STAligner import STAligner
import colorcet as cc


import pandas as pd
import numpy as np

def add_celltype_prediction(adata):
    """
    为AnnData对象添加细胞类型预测和概率
    
    Parameters:
    adata: AnnData对象
    
    Returns:
    修改后的adata对象（就地修改）
    """
    
    # 定义所有细胞类型列名
    celltype_columns = [
        'Astrocyte', 'Bergmann', 'Choroid', 'Endothelial_mural', 
        'Endothelial_stalk', 'Ependymal', 'Fibroblast', 'Golgi', 
        'Granule', 'MLI1', 'MLI2', 'Macrophage', 'Microglia', 
        'ODC', 'OPC', 'PLI', 'Purkinje', 'UBC'
    ]
    
    # 检查所有细胞类型列是否存在
    missing_columns = [col for col in celltype_columns if col not in adata.obs.columns]
    if missing_columns:
        print(f"警告: 以下细胞类型列不存在: {missing_columns}")
        celltype_columns = [col for col in celltype_columns if col in adata.obs.columns]
    print(f"使用的细胞类型列: {celltype_columns}")
    # 提取概率矩阵
    prob_matrix = adata.obs[celltype_columns].values
    # 找到每行最大值的索引
    max_indices = np.argmax(prob_matrix, axis=1)
    # 获取最大概率值
    max_probabilities = prob_matrix[np.arange(len(prob_matrix)), max_indices]
    # 获取对应的细胞类型名称
    predicted_celltypes = [celltype_columns[idx] for idx in max_indices]
    # 添加到adata.obs
    adata.obs['celltype'] = predicted_celltypes
    adata.obs['celltype_proba'] = max_probabilities
    
    # 打印统计信息
    print("\n细胞类型预测完成!")
    print(f"总细胞数: {len(adata.obs)}")
    print("\n各细胞类型数量:")
    celltype_counts = adata.obs['celltype'].value_counts()
    for celltype, count in celltype_counts.items():
        percentage = (count / len(adata.obs)) * 100
        print(f"  {celltype}: {count} ({percentage:.2f}%)")
    
    print(f"\n概率值统计:")
    print(f"  最小概率: {adata.obs['celltype_proba'].min():.6f}")
    print(f"  最大概率: {adata.obs['celltype_proba'].max():.6f}")
    print(f"  平均概率: {adata.obs['celltype_proba'].mean():.6f}")
    print(f"  中位数概率: {adata.obs['celltype_proba'].median():.6f}")
    
    return adata

# 使用示例
# adata = add_celltype_prediction(adata)

# 可选：查看结果
def show_prediction_results(adata, n_samples=10):
    """
    显示预测结果示例
    """
    print(f"\n前{n_samples}个细胞的预测结果:")
    result_df = adata.obs[['celltype', 'celltype_proba']].head(n_samples)
    for idx, (celltype, proba) in result_df.iterrows():
        print(f"  {idx}: {celltype} (概率: {proba:.6f})")

# 使用方法:
# show_prediction_results(adata)

In [2]:
data_path =  '../Experiment_cerebellar/Data/Macaque/'
section_name_list = ['Macaque1_T44'] #,, 'Macaque1_T72', 'Macaque1_T74'
for section_name in section_name_list:
    save_section_name = section_name
    section_name = section_name + '.h5ad'
    adata = sc.read_h5ad(data_path + section_name)
    print(adata)
  
    adata.obs['batch_name'] = save_section_name
    adata.obs['slice_name'] = save_section_name
    adata.obs['species_id'] = 'Macaque'

    adata_temp = adata.copy()
    sc.pp.normalize_total(adata_temp, target_sum=1e5)
    sc.pp.log1p(adata_temp)
    sc.pp.highly_variable_genes(adata_temp, n_top_genes=2000, subset=True)
    # 2. PCA 降维到 2D
    sc.tl.pca(adata_temp, n_comps=30)
    # PCA 结果存储在 adata.obsm['X_pca']，取前两列
    adata.obsm['spatial_pca'] = adata_temp.obsm['X_pca'][:, :30]
    
    adata = add_celltype_prediction(adata)
    adata.obs['annotation'] = adata.obs['celltype']
    adata.obs['region_name'] = adata.obs['region']
    print(adata)
    adata.write_h5ad('./Data/Macaque/' + save_section_name + '_celltype' + '.h5ad')

AnnData object with n_obs × n_vars = 53264 × 15744
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'region', 'coor_x', 'coor_y', 'raw_x', 'raw_y', 'distance', 'area', 'Bergmann', 'Endothelial', 'Fibroblast', 'Golgi', 'Granule', 'MLI1', 'MLI2', 'OPC', 'Purkinje', 'UBC', 'Astrocyte', 'Choroid', 'Microglia', 'ODC', 'cluster', 'annotation'
    var: 'geneID', 'features'
    uns: 'annotation_colors'
    obsm: 'spatial'
警告: 以下细胞类型列不存在: ['Endothelial_mural', 'Endothelial_stalk', 'Ependymal', 'Macrophage', 'PLI']
使用的细胞类型列: ['Astrocyte', 'Bergmann', 'Choroid', 'Fibroblast', 'Golgi', 'Granule', 'MLI1', 'MLI2', 'Microglia', 'ODC', 'OPC', 'Purkinje', 'UBC']

细胞类型预测完成!
总细胞数: 53264

各细胞类型数量:
  Granule: 11562 (21.71%)
  Purkinje: 11347 (21.30%)
  ODC: 6526 (12.25%)
  Astrocyte: 4769 (8.95%)
  Fibroblast: 4299 (8.07%)
  Bergmann: 3287 (6.17%)
  Choroid: 3114 (5.85%)
  MLI1: 2239 (4.20%)
  Microglia: 1589 (2.98%)
  MLI2: 1408 (2.64%)
  UBC: 1325 (2.49%)
  OPC: 940 (1.76%)
  Golgi: 859 (1.61%)

概率值统

In [3]:
data_path =  '../Experiment_cerebellar/Data/Mouse/'
section_name_list = ['Mouse1_T170'] #,, 'Macaque1_T72', 'Macaque1_T74'
for section_name in section_name_list:
    save_section_name = section_name
    section_name = section_name + '.h5ad'
    adata = sc.read_h5ad(data_path + section_name)
    print(adata)
  
    adata.obs['batch_name'] = save_section_name
    adata.obs['slice_name'] = save_section_name
    adata.obs['species_id'] = 'Mouse'

    adata_temp = adata.copy()
    sc.pp.normalize_total(adata_temp, target_sum=1e5)
    sc.pp.log1p(adata_temp)
    sc.pp.highly_variable_genes(adata_temp, n_top_genes=2000, subset=True)
    # 2. PCA 降维到 2D
    sc.tl.pca(adata_temp, n_comps=30)
    # PCA 结果存储在 adata.obsm['X_pca']，取前两列
    adata.obsm['spatial_pca'] = adata_temp.obsm['X_pca'][:, :30]
    
    adata = add_celltype_prediction(adata)
    adata.obs['annotation'] = adata.obs['celltype']
    adata.obs['region_name'] = adata.obs['region']
    print(adata)
    adata.write_h5ad('./Data/Mouse/' + save_section_name + '_celltype' + '.h5ad')

AnnData object with n_obs × n_vars = 35753 × 22747
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'region', 'coor_x', 'coor_y', 'raw_x', 'raw_y', 'distance', 'area', 'Astrocyte', 'Bergmann', 'Choroid', 'Endothelial_mural', 'Endothelial_stalk', 'Ependymal', 'Fibroblast', 'Golgi', 'Granule', 'MLI1', 'MLI2', 'Macrophage', 'Microglia', 'ODC', 'OPC', 'PLI', 'Purkinje', 'UBC', 'cluster', 'annotation'
    var: 'geneID', 'features'
    uns: 'annotation_colors'
    obsm: 'spatial'
使用的细胞类型列: ['Astrocyte', 'Bergmann', 'Choroid', 'Endothelial_mural', 'Endothelial_stalk', 'Ependymal', 'Fibroblast', 'Golgi', 'Granule', 'MLI1', 'MLI2', 'Macrophage', 'Microglia', 'ODC', 'OPC', 'PLI', 'Purkinje', 'UBC']

细胞类型预测完成!
总细胞数: 35753

各细胞类型数量:
  Granule: 6151 (17.20%)
  Endothelial_stalk: 4896 (13.69%)
  Ependymal: 4589 (12.84%)
  Fibroblast: 3228 (9.03%)
  Purkinje: 3007 (8.41%)
  Bergmann: 2361 (6.60%)
  Choroid: 2238 (6.26%)
  ODC: 2131 (5.96%)
  MLI2: 1302 (3.64%)
  Endothelial_mural: 1267 (3.54%)
  

In [4]:
data_path =  '../Experiment_cerebellar/Data/Marmoset/'
section_name_list = ['Marmoset1_T484'] #,, 'Macaque1_T72', 'Macaque1_T74'
for section_name in section_name_list:
    save_section_name = section_name
    section_name = section_name + '.h5ad'
    adata = sc.read_h5ad(data_path + section_name)
    print(adata)
  
    adata.obs['batch_name'] = save_section_name
    adata.obs['slice_name'] = save_section_name
    adata.obs['species_id'] = 'Marmoset'

    adata_temp = adata.copy()
    sc.pp.normalize_total(adata_temp, target_sum=1e5)
    sc.pp.log1p(adata_temp)
    sc.pp.highly_variable_genes(adata_temp, n_top_genes=2000, subset=True)
    # 2. PCA 降维到 2D
    sc.tl.pca(adata_temp, n_comps=30)
    # PCA 结果存储在 adata.obsm['X_pca']，取前两列
    adata.obsm['spatial_pca'] = adata_temp.obsm['X_pca'][:, :30]
    
    adata = add_celltype_prediction(adata)
    adata.obs['annotation'] = adata.obs['celltype']
    adata.obs['region_name'] = adata.obs['region']
    print(adata)
    adata.write_h5ad('./Data/Marmoset/' + save_section_name + '_celltype' + '.h5ad')

AnnData object with n_obs × n_vars = 27390 × 27658
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'region', 'coor_x', 'coor_y', 'raw_x', 'raw_y', 'distance', 'area', 'Astrocyte', 'Bergmann', 'Choroid', 'Endothelial_stalk', 'Ependymal', 'Fibroblast', 'Golgi', 'Granule', 'MLI1', 'MLI2', 'Microglia', 'ODC', 'OPC', 'Purkinje', 'UBC', 'cluster', 'annotation'
    var: 'geneID', 'features'
    uns: 'annotation_colors'
    obsm: 'spatial'
警告: 以下细胞类型列不存在: ['Endothelial_mural', 'Macrophage', 'PLI']
使用的细胞类型列: ['Astrocyte', 'Bergmann', 'Choroid', 'Endothelial_stalk', 'Ependymal', 'Fibroblast', 'Golgi', 'Granule', 'MLI1', 'MLI2', 'Microglia', 'ODC', 'OPC', 'Purkinje', 'UBC']

细胞类型预测完成!
总细胞数: 27390

各细胞类型数量:
  Granule: 8030 (29.32%)
  Purkinje: 4740 (17.31%)
  Bergmann: 3150 (11.50%)
  ODC: 2146 (7.83%)
  Golgi: 1469 (5.36%)
  Fibroblast: 1414 (5.16%)
  MLI1: 1354 (4.94%)
  Astrocyte: 1315 (4.80%)
  Endothelial_stalk: 1057 (3.86%)
  Microglia: 860 (3.14%)
  Ependymal: 684 (2.50%)
  MLI2: 414 (